In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as si
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")
import QuantLib as ql
import plotly.graph_objects as go
import os
import regex as re
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy import interpolate
from scipy.optimize import minimize
import seaborn as sns
from tqdm import tqdm
from datetime import datetime, timedelta

In [ ]:
#这块是提取要用的数据
data_path = "2years_5pct.csv"

tmp = pd.read_csv(data_path)

# 确保 QUOTE_DATE 列为 datetime64 类型
#tmp['QUOTE_DATE'] = pd.to_datetime(tmp['QUOTE_DATE'])

# 筛选 2023 2022 2021 2020 2019 2018 2017 2016 2015 2014年的数据
#tmp = tmp[tmp['QUOTE_DATE'].dt.year == 2023]
#tmp = tmp[tmp['DTE'] == 365 * 2].reset_index(drop=True)

tmp

In [ ]:
#getting ssvi iv
def compute_theta_t(params, t, kappa1, kappa2):
    V, V_prime, theta = params
    theta_t = theta * t + (V - theta) * (1 - np.exp(-kappa1 * t)) / kappa1 + \
              (V_prime - theta) * kappa1 * ((1 - np.exp(-kappa2 * t)) / kappa2 - \
              (1 - np.exp(-kappa1 * t)) / kappa1) / (kappa1 - kappa2)
    return theta_t

def compute_phi(params_w_t, params_theta_t, t, phi_type, kappa1, kappa2):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t, kappa1, kappa2)
    if phi_type == "POWER-LAW1":
        return eta / theta_t ** gamma
    elif phi_type == "POWER-LAW2":
        return eta / (theta_t ** gamma * (1 + theta_t) ** (1 - gamma))
    elif phi_type == "SQRT":
        return eta / np.sqrt(theta_t)
    elif phi_type == "HESTON":
        return (1 - (1 - np.exp(-gamma * theta_t)) / (gamma * theta_t)) / (gamma * theta_t)
    else:
        raise ValueError("Invalid phi_type")

def compute_ssvi_inference(params_w_t, params_theta_t, t, k, phi_type, kappa1, kappa2):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t, kappa1, kappa2)
    phi = compute_phi(params_w_t, params_theta_t, t, phi_type, kappa1, kappa2)
    return 0.5 * theta_t * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + 1 - rho ** 2))


# 定义 kappa1 和 kappa2
kappa1 = 5.5
kappa2 = 0.1

# 使用 apply 方法，传递所有必要参数
tmp['ssvi_iv'] = tmp.apply(
    lambda row: np.sqrt(
        compute_ssvi_inference(
            params_w_t=[row['rho'], row['eta'], row['gamma']],
            params_theta_t=[row['V'], row['V_prime'], row['theta']],
            t=2,
            k=row['Log-Strike'],
            phi_type="POWER-LAW1",
            kappa1=kappa1,
            kappa2=kappa2
        ) / 2
    ), axis=1
)

tmp

In [ ]:
a = tmp['ssvi_iv'].min()
b = tmp['ssvi_iv'].max()
print(a)
print(b)

In [ ]:

#计算binomial美式价格
def binomial_tree_american_option(S0, K, T, r, sigma, N, q, option_type="call"):
    """
    Prices an American option using the binomial tree model with dividends.
    
    Parameters:
    S0 : float - Initial stock price
    K : float - Strike price
    T : float - Time to maturity in years
    r : float - Risk-free interest rate
    sigma : float - Volatility
    N : int - Number of steps in the binomial tree
    q : float - Dividend yield
    option_type : str - Type of the option ("call" or "put")
    
    Returns:
    float - Estimated option price
    """  
    dt = T / N
    u = np.exp((r - q) * dt + sigma * np.sqrt(dt))
    d = np.exp((r - q) * dt - sigma * np.sqrt(dt))
    p = (np.exp((r - q) * dt) - d) / (u - d)

    # 初始化股票价格树
    stock_prices = np.asarray([S0 * u**j * d**(N-j) for j in range(N+1)])
    
    # 初始化期权价值树的最终节点
    if option_type == "call":
        option_values = np.maximum(stock_prices - K, 0)
    else:  # put
        option_values = np.maximum(K - stock_prices, 0)

    # 反向迭代期权价值树
    for i in range(N-1, -1, -1):
        stock_prices = S0 * u**np.arange(i+1) * d**(i-np.arange(i+1))
        option_values_early = np.maximum(stock_prices - K, 0) if option_type == "call" else np.maximum(K - stock_prices, 0)
        option_values = np.exp(-r * dt) * (p * option_values[1:] + (1 - p) * option_values[:-1])
        option_values = np.maximum(option_values, option_values_early)  # 早期行权的检查

    return option_values[0]

# 设置二叉树的步数
#N = 100  # 或者根据需要调整

############################################################################################################

# def bisect(f, a, b, tol):
#     assert a < b and f(a) * f(b) < 0, "The assumptions of the bisection method are not met."
#     c = (a + b) / 2
#     while (b - a) / 2 > tol:
#         c = (a + b) / 2
#         if abs(f(c)) < tol:
#             return c
#         else:
#             if f(a) * f(c) < 0:
#                 b = c
#             else:
#                 a = c
#     return c

def bisect(f, a, b, tol):
    if f(a) * f(b) >= 0:
        return None  # 如果a和b之间不存在根，则返回None
    c = (a + b) / 2
    while (b - a) / 2 > tol:
        if abs(f(c)) < tol:
            return c
        elif f(a) * f(c) < 0:
            b = c
        else:
            a = c
        c = (a + b) / 2
    return c

############################################################################################################

def amer_implied_vol(S0, r, T, K, q, market_price, option_type="call"):
    """
    Bisection algorithm to find the implied American volatility using the bisect function.
    """
    # 定义函数f，它计算给定波动率下的期权价格与市场价格之差
    f = lambda sigma: binomial_tree_american_option(S0, K, T, r, sigma, N=100, q=q, option_type=option_type) - market_price

    sigma_min = 0.01
    sigma_max = 10.00
    eps = 1e-5

    # 使用bisect函数查找隐含波动率
    sigma_iv = bisect(f, sigma_min, sigma_max, eps)
    
    # 返回找到的隐含波动率
    return sigma_iv

In [ ]:
#算european
def binomial_tree_european_option(S0, K, T, r, sigma, N, q, option_type="call"):
    """
    Prices a European option using the binomial tree model with dividends.
    
    Parameters:
    S0 : float - Initial stock price
    K : float - Strike price
    T : float - Time to maturity in years
    r : float - Risk-free interest rate
    sigma : float - Volatility
    N : int - Number of steps in the binomial tree
    q : float - Dividend yield
    option_type : str - Type of the option ("call" or "put")
    
    Returns:
    float - Estimated option price
    """  
    dt = T / N
    u = np.exp((r - q) * dt + sigma * np.sqrt(dt))
    d = np.exp((r - q) * dt - sigma * np.sqrt(dt))
    p = (np.exp((r - q) * dt) - d) / (u - d)

    # 初始化股票价格树
    stock_prices = np.asarray([S0 * u**j * d**(N-j) for j in range(N+1)])
    
    # 初始化期权价值树的最终节点
    if option_type == "call":
        option_values = np.maximum(stock_prices - K, 0)
    else:  # put
        option_values = np.maximum(K - stock_prices, 0)

    # 反向迭代期权价值树，不检查早期行权
    for i in range(N-1, -1, -1):
        option_values = np.exp(-r * dt) * (p * option_values[1:] + (1 - p) * option_values[:-1])

    return option_values[0]


In [ ]:
print(tmp.iloc[284])

In [ ]:
import pandas as pd

# implied_vols_put = []
# implied_vols_call = []

# # 假设 amer_implied_vol 是之前定义的函数，用于计算隐含波动率

# # 遍历前 100 行数据
# for index, row in tmp.iloc[:100].iterrows():
#     # 计算看跌期权的隐含波动率
#     implied_vol_put = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_put'], option_type="put")
#     # 计算看涨期权的隐含波动率
#     implied_vol_call = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_call'], option_type="call")
#     # 将隐含波动率值添加到相应的列表中
#     implied_vols_put.append(implied_vol_put)
#     implied_vols_call.append(implied_vol_call)

# # 将隐含波动率列表添加为 tmp DataFrame 的新列
# tmp['implied_vol_put'] = pd.NA  # 初始化列为缺失值
# tmp['implied_vol_call'] = pd.NA  # 初始化列为缺失值
# tmp.loc[tmp.index[:100], 'implied_vol_put'] = implied_vols_put  # 填充前 100 行
# tmp.loc[tmp.index[:100], 'implied_vol_call'] = implied_vols_call  # 填充前 100 行

# 初始化两个空列表以存储隐含波动率
tmp['implied_vol_put'] = pd.NA
tmp['implied_vol_call'] = pd.NA

# 遍历DataFrame中的所有行并尝试计算隐含波动率
for index, row in tmp.iterrows():
    try:
        implied_vol_put = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_put'], option_type="put")
        implied_vol_call = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_call'], option_type="call")
    except AssertionError:
        print(f"Index {index}: The assumptions of the bisection method are not met.")
        implied_vol_put = None
        implied_vol_call = None
    
    tmp.at[index, 'implied_vol_put'] = implied_vol_put if implied_vol_put is not None else pd.NA
    tmp.at[index, 'implied_vol_call'] = implied_vol_call if implied_vol_call is not None else pd.NA

tmp

In [ ]:
tmp.to_csv('simuvol.csv', index=False)

In [ ]:
for index, row in tmp.iterrows():
    # 仅当索引大于等于 285 时，执行以下操作
    if index >= 500:
        # 计算看跌期权的隐含波动率
        implied_vol_put = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_put'], option_type="put")
        # 计算看涨期权的隐含波动率
        implied_vol_call = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 0, row['amer_call'], option_type="call")
        print(f"Index {index}: Implied Vol Put = {implied_vol_put}, Implied Vol Call = {implied_vol_call}")
        # 直接在DataFrame中更新隐含波动率值
        tmp.at[index, 'implied_vol_put'] = implied_vol_put
        tmp.at[index, 'implied_vol_call'] = implied_vol_call

In [ ]:
rows = tmp.iloc[:100].copy()

def calculate_option_prices(row, option_type):
    S0 = row['S']
    K = row['strike']
    T = 2
    r = row['r']
    sigma = row['implied_vol_put'] if option_type == 'put' else row['implied_vol_call']
    N = 1000  # 步数
    q = 0  #设为0

    option_price = binomial_tree_european_option(S0, K, T, r, sigma, N, q, option_type)

    return option_price

# 计算看跌期权价格并添加到新列
rows['new_europut'] = rows.apply(lambda row: calculate_option_prices(row, 'put'), axis=1)

# 计算看涨期权价格并添加到新列
rows['new_eurocall'] = rows.apply(lambda row: calculate_option_prices(row, 'call'), axis=1)

In [ ]:
rows

In [ ]:
# 计算隐含波动率与ssvi_iv的差异 timestep1000
rows['diff_put2'] = rows['implied_vol_put'] - rows['ssvi_iv']
rows['diff_call2'] = rows['implied_vol_call'] - rows['ssvi_iv']

# 将新计算的差异列添加到 diff_df 中
df['diff_put2'] = rows['diff_put2']
df['diff_call2'] = rows['diff_call2']
df['strike']=rows['strike']
rows

In [ ]:
# 计算隐含波动率与ssvi_iv的差异
rows['diff_put'] = rows['implied_vol_put'] - rows['ssvi_iv']
rows['diff_call'] = rows['implied_vol_call'] - rows['ssvi_iv']


plt.figure(figsize=(10, 6))

plt.plot(rows['diff_put'], label='Diff Implied Vol Put - SSVI IV', color='red', linestyle='--')
plt.plot(rows['diff_call'], label='Diff Implied Vol Call - SSVI IV', color='green', linestyle='-')

plt.title('Difference Between Implied Volatility and SSVI IV')
plt.xlabel('Index')
plt.ylabel('Difference')
plt.legend()

plt.show()
df = rows[['diff_put', 'diff_call']].copy()

In [ ]:
df

In [ ]:
# 绘制看跌期权隐含波动率差异图
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(df['strike'], df['diff_put'], label='Diff Put', color='blue', marker='o')
plt.plot(df['strike'], df['diff_put2'], label='Diff Put2', color='cyan', linestyle='--', marker='x')
plt.title('Difference in Implied Volatility for Put Options')
plt.xlabel('Strike Price')
plt.ylabel('Difference')
plt.legend()
plt.grid(True)

# 绘制看涨期权隐含波动率差异图
plt.subplot(1, 2, 2)
plt.plot(df['strike'], df['diff_call'], label='Diff Call', color='red', marker='o')
plt.plot(df['strike'], df['diff_call2'], label='Diff Call2', color='orange', linestyle='--', marker='x')
plt.title('Difference in Implied Volatility for Call Options')
plt.xlabel('Strike Price')
plt.ylabel('Difference')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# 计算差值
df['diff_put_difference'] = df['diff_put'] - df['diff_put2']
df['diff_call_difference'] = df['diff_call'] - df['diff_call2']

# 绘制差值图
plt.figure(figsize=(12, 6))

# 绘制 diff_put 与 diff_put2 的差值图
plt.subplot(1, 2, 1)
plt.plot(df['strike'], df['diff_put_difference'], label='Diff Put - Diff Put2', color='blue', marker='o')
plt.title('Difference Between Diff Put and Diff Put2')
plt.xlabel('Strike Price')
plt.ylabel('Difference')
plt.legend()
plt.grid(True)

# 绘制 diff_call 与 diff_call2 的差值图
plt.subplot(1, 2, 2)
plt.plot(df['strike'], df['diff_call_difference'], label='Diff Call - Diff Call2', color='red', marker='o')
plt.title('Difference Between Diff Call and Diff Call2')
plt.xlabel('Strike Price')
plt.ylabel('Difference')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
#call
# 获取所需的数据
strike_prices = rows['strike']
new_euro_call_prices = rows['new_eurocall']
our_call_prices = rows['eur_call']

# 绘制图表
plt.figure(figsize=(10, 6))
plt.plot(strike_prices, new_euro_call_prices, label='New Euro Call')
plt.plot(strike_prices, our_call_prices, label='Euro Call')
plt.xlabel('Strike Price')
plt.ylabel('Option Price')
plt.title('Comparison of New Euro Call and Market Euro Call (T=2)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
#put
# 获取所需的数据
strike_prices = rows['strike']
new_euro_call_prices = rows['new_europut']
our_call_prices = rows['eur_put']

# 绘制图表
plt.figure(figsize=(10, 6))
plt.plot(strike_prices, new_euro_call_prices, label='New Euro Put')
plt.plot(strike_prices, our_call_prices, label='Euro Put')
plt.xlabel('Strike Price')
plt.ylabel('Option Price')
plt.title('Comparison of New Euro Put and Market Euro Put (T=2)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#put
difference = rows['new_europut'] - rows['eur_put']

# 获取所需的数据
strike_prices = rows['strike']

# 绘制差值图表
plt.figure(figsize=(10, 6))
plt.plot(strike_prices, difference, label='Difference (New Euro put - Market put)', color='red')
plt.xlabel('Strike Price')
plt.ylabel('Difference in Option Price')
plt.title('Difference between New Euro Put and Market Put (T=2)')
plt.axhline(y=0, color='black', linestyle='--')  # 添加水平参考线，表示差值为0
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
#call
difference = rows['new_eurocall'] - rows['eur_call']

# 获取所需的数据
strike_prices = rows['strike']

# 绘制差值图表
plt.figure(figsize=(10, 6))
plt.plot(strike_prices, difference, label='Difference (New Euro call - Market call)', color='red')
plt.xlabel('Strike Price')
plt.ylabel('Difference in Option Price')
plt.title('Difference between New Euro Call and Market Call (T=2)')
plt.axhline(y=0, color='black', linestyle='--')  # 添加水平参考线，表示差值为0
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
rows

In [ ]:
#计算binomial美式价格
def binomial_tree_american_option(S0, K, T, r, sigma, N, q, option_type="call"):
    """
    Prices an American option using the binomial tree model with dividends.
    
    Parameters:
    S0 : float - Initial stock price
    K : float - Strike price
    T : float - Time to maturity in years
    r : float - Risk-free interest rate
    sigma : float - Volatility
    N : int - Number of steps in the binomial tree
    q : float - Dividend yield
    option_type : str - Type of the option ("call" or "put")
    
    Returns:
    float - Estimated option price
    """  
    dt = T / N
    u = np.exp((r - q) * dt + sigma * np.sqrt(dt))
    d = np.exp((r - q) * dt - sigma * np.sqrt(dt))
    p = (np.exp((r - q) * dt) - d) / (u - d)

    # 初始化股票价格树
    stock_prices = np.asarray([S0 * u**j * d**(N-j) for j in range(N+1)])
    
    # 初始化期权价值树的最终节点
    if option_type == "call":
        option_values = np.maximum(stock_prices - K, 0)
    else:  # put
        option_values = np.maximum(K - stock_prices, 0)

    # 反向迭代期权价值树
    for i in range(N-1, -1, -1):
        stock_prices = S0 * u**np.arange(i+1) * d**(i-np.arange(i+1))
        option_values_early = np.maximum(stock_prices - K, 0) if option_type == "call" else np.maximum(K - stock_prices, 0)
        option_values = np.exp(-r * dt) * (p * option_values[1:] + (1 - p) * option_values[:-1])
        option_values = np.maximum(option_values, option_values_early)  # 早期行权的检查

    return option_values[0]

# 设置二叉树的步数
#N = 100  # 或者根据需要调整

# 重新定义计算期权价格的函数以考虑股息
def calc_option_price(row, option_type="call"):
    return binomial_tree_american_option(
        S0=row['S'], 
        K=row['strike'], 
        T=2,  # 假设DTE是以天为单位，转换为年
        r=row['r'], 
        sigma=row['ssvi_iv'], 
        N=100,
        q=0,  # 考虑股息
        option_type=option_type
    )

import sys  # for sys.exit, consider replacing with exception handling

def amer_implied_vol(S0, r, T, K, q, market_price, option_type="call"):
    """
    Bisection algorithm to find the implied American volatility.
    """
    sigma_min = 0.001
    sigma_max = 10
    n_max = 750
    eps = 1e-5

    n = 1
    while n <= n_max:
        sigma = (sigma_min + sigma_max) / 2
        f_sigma = binomial_tree_american_option(S0, K, T, r, sigma, 100, q, option_type) - market_price

        if abs(f_sigma) < eps or (sigma_max - sigma_min) < eps:
            return sigma
        if f_sigma < 0:
            sigma_min = sigma
        else:
            sigma_max = sigma
        n += 1

    raise Exception(f"The American volatility calibration algorithm failed to converge for the {option_type.upper()} option with Strike {K}")
    


    
# Example usage with a row from df_merged_tmp_ssvi
row = df_merged_tmp_ssvi.iloc[0]  # Example row
implied_vol = amer_implied_vol(row['S'], row['r'], 2, row['strike'], 'call', 0, row['amer_call'])



In [ ]:
df_merged_tmp_ssvi['implied_vol_call'] = df_merged_tmp_ssvi.apply(
    lambda row: amer_implied_vol(
        S0=row['S'], 
        r=row['r'], 
        T=2,  # Assuming 2 years to expiration
        K=row['strike'], 
        option_type='call', 
        q=0,  # Assuming no dividends for simplicity, adjust as needed
        market_price=row['amer_call']
    ), axis=1)

In [ ]:
df_merged_tmp_ssvi